# Baseline 2 — Exp06: Seen-Pair Score Calibration (Development Only)

**Purpose:** Test whether down-weighting *training-seen crop/disease pair scores* improves compositional generalization, **without retraining or changing the B2 architecture**. This is a standard **calibrated-stacking-inspired post-processing experiment**, *not a newly trained model*. Keep the original uncalibrated Exp04 results as the canonical B2 baseline; report Exp06 separately as **B2 + calibration**.

**Strict boundaries:** Uses only the saved **Exp04 Stage 1 development checkpoint** and the three original development CSVs (`train_fit`, `val_seen`, `val_unseen`). No official test inputs, no Stage 1/2 training, no changes to weights. Candidate policy remains **all 9,625 crop×disease combinations**. `gamma=0` must reproduce Exp04.

Paper for report: Chao et al. (2016), *An Empirical Study and Analysis of Generalized Zero-Shot Learning for Object Recognition in the Wild*, ECCV. In the CZSL setting, apply the bias at the **pair level**, where seen pairs are identified **only from train_fit**, not from the validation labels.


This reference copy has inference disabled by default.

## 1. Imports and safety controls
Place this notebook **alongside** your existing `model.py`, `evaluate.py`, and `splits_v2/`. Keep `RUN_EXP06 = False` until all checks and definitions have run. **Do not use Run All with training switches from an older notebook.**

In [ ]:
from pathlib import Path
import json
import os
import sys
import time
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
# Locate the project before importing its modules, including from nested notebook folders.
kernel_dir = Path.cwd().resolve()
PROJECT_DIR = next(
    (folder for folder in (kernel_dir, *kernel_dir.parents)
     if all((folder / name).is_file() for name in ("baseline2.ipynb", "model.py", "evaluate.py"))),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Start Jupyter in the Baseline2 folder or one of its subfolders.")
sys.path.insert(0, str(PROJECT_DIR))

from model import load_checkpoint
from evaluate import (
    DPDDataset, validate_development_splits, validation_transform,
    composition_scores, make_candidates, harmonic_mean,
    require_candidate_coverage, sha256_file
)

RUN_EXP06 = False

SPLIT_DIR = PROJECT_DIR / "splits_v2"
IMAGE_ROOT = Path(os.environ.get(
    "B2_IMAGE_ROOT", str(PROJECT_DIR.parent / "DPD_Small_dataset")
)).expanduser()
IMAGE_ROOT = (PROJECT_DIR / IMAGE_ROOT).resolve()
CHECKPOINT_PATH = PROJECT_DIR / "outputs" / "exp04" / "best_development.pt"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "exp06_calibration"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Predetermined, small sweep (score units = sum of log-probabilities).
GAMMA_GRID = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]
BATCH_SIZE = 32
NUM_WORKERS = 0  # Stable on Windows Jupyter; no training, speed not critical.

assert (PROJECT_DIR / "model.py").is_file(), "Missing model.py: check PROJECT_DIR"
assert (PROJECT_DIR / "evaluate.py").is_file(), "Missing evaluate.py: check PROJECT_DIR"
assert GAMMA_GRID[0] == 0.0 and len(set(GAMMA_GRID)) == len(GAMMA_GRID)
print("Device:", DEVICE, "| Exp04 checkpoint:", CHECKPOINT_PATH)

## 2. Load and audit Exp04 + development split
This block verifies the saved split file hashes against the Exp04 checkpoint to avoid silently evaluating on changed partitions.

In [ ]:
if RUN_EXP06:
    frames, split_report = validate_development_splits(SPLIT_DIR)
    model, ck = load_checkpoint(CHECKPOINT_PATH, DEVICE)
    assert ck.get("stage") == 1, "Expect Stage 1 development checkpoint, not final model"
    assert str(ck.get("config", {}).get("experiment_id", "exp04")) == "exp04", "Checkpoint is not Exp04"
    assert split_report["sha256"] == ck["development_split"]["sha256"], "Development splits differ from checkpoint!"
    pair_candidates = torch.as_tensor(ck["candidate_pairs"], dtype=torch.long)
    expected_candidates = make_candidates("all_pairs")
    assert len(pair_candidates) == 9625 and torch.equal(pair_candidates, expected_candidates), "Candidate policy/order differs from all_pairs"
    require_candidate_coverage(pair_candidates, [frames["train_fit"],frames["val_seen"],frames["val_unseen"]])

    train_pairs = set(map(tuple, frames["train_fit"][["crop_id","disease_id"]].to_numpy(dtype=int).tolist()))
    seen_pair_mask = torch.as_tensor([tuple(p) in train_pairs for p in pair_candidates.tolist()], dtype=torch.bool)
    assert 0 < seen_pair_mask.sum() < 9625
    assert all(tuple(p) not in train_pairs for p in frames["val_unseen"][["crop_id","disease_id"]].to_numpy(dtype=int).tolist())
    assert all(tuple(p) in train_pairs for p in frames["val_seen"][["crop_id","disease_id"]].to_numpy(dtype=int).tolist())
    print(f"Train-seen candidate pairs: {seen_pair_mask.sum().item():,} / 9,625")
    print(f"Development splits: {len(frames['train_fit']):,} train, {len(frames['val_seen'])} seen-val, {len(frames['val_unseen'])} unseen-val")
    print("Masks derived ONLY from train_fit.csv; no validation/test-label-dependent candidate selection.")

## 3. Synthetic scoring test (no dataset, no training)
Calibrated pair score: **score(c,d | image) = log P(c | image) + log P(d | image) − γ × I[(c,d) in train_fit]**. Every 9,625 candidate remains eligible, including when γ > 0.

In [ ]:
def score_with_calibration(raw_pair_scores, seen_mask, gamma):
    if raw_pair_scores.ndim != 2 or raw_pair_scores.shape[1] != len(seen_mask):
        raise ValueError("Scores and candidate mask shape mismatch")
    if gamma < 0 or not np.isfinite(gamma):
        raise ValueError("gamma must be finite and nonnegative")
    return raw_pair_scores - float(gamma) * seen_mask.to(raw_pair_scores.dtype)[None, :]

fake = torch.tensor([[0.9, 0.8, 0.1]], dtype=torch.float32)
mask = torch.tensor([True, False, False])
assert torch.equal(score_with_calibration(fake, mask, 0.0), fake)
assert score_with_calibration(fake, mask, 0.2).argmax(1).item() == 1
assert score_with_calibration(fake, mask, 0.0).argmax(1).item() == 0
print("Synthetic calibration check passed: gamma=0 unchanged; gamma>0 penalizes only train-seen pairs.")

## 4. Define development inference (no gradients) and metric sweep
Run these function definitions once; they do **not** start inference or training.

In [ ]:
@torch.inference_mode()
def collect_pair_scores(split_name):
    dataset = DPDDataset(frames[split_name], IMAGE_ROOT, validation_transform())
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
    model.eval()
    all_scores, all_targets, all_filenames = [], [], []
    cand_dev = pair_candidates.to(DEVICE)
    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            outputs = model(images)
        raw_scores = composition_scores(outputs["crop_logits"], outputs["disease_logits"], cand_dev)
        all_scores.append(raw_scores.cpu().float())
        all_targets.append(torch.stack([batch["crop"], batch["disease"]], dim=1).cpu().long())
        all_filenames.extend(batch["filename"])
    scores = torch.cat(all_scores)
    targets = torch.cat(all_targets)
    assert scores.shape == (len(dataset),9625) and targets.shape == (len(dataset),2)
    return scores,targets,all_filenames

def evaluate_gamma(scores, targets, gamma):
    adjusted = score_with_calibration(scores,seen_pair_mask,gamma)
    top5_ids = adjusted.topk(5,dim=1).indices
    top5_pairs = pair_candidates[top5_ids]
    matches = (top5_pairs == targets[:,None,:]).all(dim=2)
    top1_acc = float(matches[:,0].float().mean())
    top5_acc = float(matches.any(dim=1).float().mean())
    predicted_seen = float(seen_pair_mask[top5_ids[:,0]].float().mean())
    return {"top1":top1_acc,"top5":top5_acc,"predicted_train_seen_pair_rate":predicted_seen},top5_pairs[:,0]

def evaluate_sweep(seen_scores,seen_targets,unseen_scores,unseen_targets):
    rows=[]
    for gamma in GAMMA_GRID:
        sm,_=evaluate_gamma(seen_scores,seen_targets,gamma)
        um,_=evaluate_gamma(unseen_scores,unseen_targets,gamma)
        rows.append({"gamma":float(gamma),"seen_top1":sm["top1"],"seen_top5":sm["top5"],
                     "unseen_top1":um["top1"],"unseen_top5":um["top5"],
                     "harmonic_mean":harmonic_mean(sm["top1"],um["top1"]),
                     "seen_predicted_train_pair_rate":sm["predicted_train_seen_pair_rate"],
                     "unseen_predicted_train_pair_rate":um["predicted_train_seen_pair_rate"]})
    results=pd.DataFrame(rows)
    # Choose the highest dev HM; ties go to smaller gamma (including 0).
    selected=results.sort_values(["harmonic_mean","gamma"],ascending=[False,True]).iloc[0]
    return results,selected

print("Inference functions ready. They don't train or update weights.")

## 5. Development inference control

RUN_EXP06 defaults to False in the configuration cell. The historical workflow requires the local Exp04 checkpoint and original development splits, and refuses to overwrite the saved experiment directory.

In [ ]:
if RUN_EXP06:
    if OUTPUT_DIR.exists():
        raise FileExistsError(f"Refusing to overwrite existing results: {OUTPUT_DIR}")
    start = time.perf_counter()
    print("Collecting Exp04 raw scores on val_seen (695), val_unseen (400) ...")
    s_scores,s_targets,s_files=collect_pair_scores("val_seen")
    u_scores,u_targets,u_files=collect_pair_scores("val_unseen")
    sweep,best=evaluate_sweep(s_scores,s_targets,u_scores,u_targets)
    base=sweep.iloc[0]
    saved_metrics=PROJECT_DIR / "outputs" / "exp04" / "validation_metrics.json"
    if saved_metrics.is_file():
        summary=json.loads(saved_metrics.read_text(encoding="utf-8"))
        exp_seen=float(summary["seen"]["composition_top1"])
        exp_unseen=float(summary["unseen"]["composition_top1"])
        assert abs(base.seen_top1-exp_seen)<1e-5 and abs(base.unseen_top1-exp_unseen)<1e-5, (
            "gamma=0 does not reproduce saved Exp04; investigate before proceeding")
    else:
        print("NOTE: Original exp04/validation_metrics.json missing. Verify baseline gamma=0 manually!")
    OUTPUT_DIR.mkdir(parents=True,exist_ok=False)
    sweep.to_csv(OUTPUT_DIR / "gamma_sweep.csv",index=False)
    report={"experiment":"exp06_calibration", "parent":"exp04", "stage":"development_postprocessing_only",
            "candidate_mode":"all_pairs","candidate_count":9625,
            "train_seen_pair_count":int(seen_pair_mask.sum()), "gamma_grid":GAMMA_GRID,
            "baseline_gamma0":base.to_dict(),"selected_on_development":best.to_dict(),
            "split_sha256":split_report["sha256"],"checkpoint_sha256":sha256_file(CHECKPOINT_PATH),
            "training_performed":False,"official_test_accessed":False,
            "selection_caveat":"Best gamma tuned on same development validation; not an unbiased final-test estimate."}
    (OUTPUT_DIR / "calibration_summary.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
    for label,scores,targets,files in [("val_seen",s_scores,s_targets,s_files),("val_unseen",u_scores,u_targets,u_files)]:
        _,pairs=evaluate_gamma(scores,targets,float(best.gamma))
        pred=pairs.numpy(); real=targets.numpy()
        pd.DataFrame({"filename":files,"true_crop_id":real[:,0],"true_disease_id":real[:,1],
                      "pred_crop_id":pred[:,0],"pred_disease_id":pred[:,1],
                      "correct":np.all(real==pred,axis=1),
                      "pred_is_train_seen_pair":[tuple(t) in train_pairs for t in pred.tolist()]})          .to_csv(OUTPUT_DIR / f"{label}_calibrated_predictions.csv",index=False)
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    axes[0].plot(sweep.gamma,sweep.seen_top1*100,marker="o",label="Seen")
    axes[0].plot(sweep.gamma,sweep.unseen_top1*100,marker="o",label="Unseen")
    axes[0].set(xlabel="Gamma",ylabel="Composition Top-1 (%)",title="Development accuracy trade-off")
    axes[0].legend()
    axes[1].plot(sweep.gamma,sweep.harmonic_mean*100,marker="o")
    axes[1].axvline(float(best.gamma),ls="--",label=f"Best gamma {best.gamma:g}")
    axes[1].set(xlabel="Gamma",ylabel="Harmonic Mean (%)",title="Development selection")
    axes[1].legend()
    fig.tight_layout();fig.savefig(OUTPUT_DIR / "calibration_curve.png",dpi=170)
    plt.show();plt.close(fig)
    print("\nBaseline (gamma=0):",{k:round(float(base[k])*100,3) for k in ("seen_top1","unseen_top1","harmonic_mean")})
    print("Selected gamma:",float(best.gamma),"| Results:",
          {k:round(float(best[k])*100,3) for k in ("seen_top1","unseen_top1","harmonic_mean")})
    print("Completed in",round(time.perf_counter()-start,1),"seconds. Results:",OUTPUT_DIR)
else:
    print("Exp06 OFF — no inference or training. Set RUN_EXP06=True ONLY in this cell to run.")

## 6. Recorded result

The selected development penalty was γ = 4. Harmonic mean increased from 4.35% to 11.40%, with a decrease in seen Top-1. The official calibrated mode used the same penalty and the mask derived from all 8,000 training images.

See the [saved Exp06 report](../../outputs/development/exp06/calibration_summary.json).